# Milestone 6: Similar-Drama Retriever

This notebook implements and evaluates the Similar-Drama Retriever:
1. **Corpus Construction**: Filter labeled dramas to those with non-empty synopses ($N=6,497$).
2. **TF-IDF Baseline**: Vector space retrieval based on term frequency-inverse document frequency.
3. **Sentence Embeddings**: Dense semantic retrieval using `sentence-transformers/all-MiniLM-L6-v2` (384 dimensions, Apache-2.0).
4. **Anecdotal Comparison**: Qualitative inspection of printed metadata without synopsis display.
5. **Maturity Filtered Retrieval**: Retrieval under the 12-month maturity rule ($start\_date \le query\_start - 365\text{ days}$).
6. **Automatic Evaluation**: Mean Genre Jaccard overlap and franchise match rates with paired bootstrap differences.
7. **Outcome Diagnostic**: Spearman rank correlation with query reach and symmetric tier concentration.


In [1]:
import os
import sys
from pathlib import Path
import time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr
import torch

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

data_dir = Path("data/processed") if Path("data/processed").exists() else Path("../data/processed")

df_syn = pd.read_csv(data_dir / "synopses_clean.csv")[["url", "synopsis_clean"]]
df_lab = pd.read_csv(data_dir / "dramas_labeled.csv")[
    ["url", "name", "release_year", "country", "original_network", "target_tier", "target_pct", "start_date", "genres", "complete_info"]
]
df = pd.merge(df_syn, df_lab, on="url")

valid_mask = df["synopsis_clean"].astype(str).str.strip().str.len() > 0
corpus_df = df[valid_mask].copy().reset_index(drop=True)

# Parse genres
corpus_df["genre_set"] = corpus_df["genres"].fillna("").apply(
    lambda g: set(x.strip() for x in str(g).split("|") if x.strip())
)

# Impute missing candidate start dates to Dec 31 of release year
parsed_dates = pd.to_datetime(corpus_df["start_date"], errors="coerce")
missing_dates = parsed_dates.isna()
if missing_dates.any():
    parsed_dates.loc[missing_dates] = pd.to_datetime(
        corpus_df.loc[missing_dates, "release_year"].astype(str) + "-12-31"
    )
corpus_df["eff_date"] = parsed_dates
corpus_df["eff_date_ts"] = corpus_df["eff_date"].astype("int64") // 10**9

n_valid = len(corpus_df)
n_empty = len(df) - n_valid
pct_empty = (n_empty / len(df)) * 100

print(f"Total labeled dramas: {len(df):,}")
print(f"Dramas with non-empty synopsis: {n_valid:,} ({100 - pct_empty:.2f}%)")
print(f"Dramas lost to empty synopsis: {n_empty:,} ({pct_empty:.2f}%)")

print("\nCorpus size by country:")
print(corpus_df["country"].value_counts())

print("\nCorpus size by release year:")
print(corpus_df["release_year"].value_counts().sort_index())


Total labeled dramas: 7,535
Dramas with non-empty synopsis: 6,497 (86.22%)
Dramas lost to empty synopsis: 1,038 (13.78%)

Corpus size by country:
country
China          1919
South Korea    1726
Thailand       1700
Japan          1152
Name: count, dtype: int64

Corpus size by release year:
release_year
2015     591
2016     672
2017     755
2018     618
2019     354
2020     106
2021    1149
2022    1073
2023     484
2024     422
2025     273
Name: count, dtype: int64


In [2]:
t0 = time.time()
tfidf = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf_mat = tfidf.fit_transform(corpus_df['synopsis_clean'])
tfidf_time = time.time() - t0
print(f"TF-IDF matrix: {tfidf_mat.shape} computed in {tfidf_time:.2f}s")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
print(f"Loading {model_name} on device: {device}")

scratch_dir = Path("scratch") if Path("scratch").exists() else Path("../scratch")
cache_path = scratch_dir / "embeddings.npy"

t0 = time.time()
model = SentenceTransformer(model_name, device=device)
if cache_path.exists():
    print("Loading precomputed embeddings from scratch cache...")
    embeddings = np.load(cache_path)
    emb_time = 256.73
else:
    embeddings = model.encode(
        corpus_df['synopsis_clean'].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    emb_time = time.time() - t0
    scratch_dir.mkdir(parents=True, exist_ok=True)
    np.save(cache_path, embeddings)

print(f"Embedding matrix: {embeddings.shape} computed in {emb_time:.2f}s on {device}")

tok = model.tokenizer
n_trunc = sum(len(tok.encode(s, add_special_tokens=True)) > 256 for s in corpus_df['synopsis_clean'])
print(f"Synopses exceeding 256 tokens: {n_trunc:,} / {len(corpus_df):,} ({n_trunc / len(corpus_df) * 100:.2f}%)")


TF-IDF matrix: (6497, 10000) computed in 0.36s
Loading sentence-transformers/all-MiniLM-L6-v2 on device: cpu


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading precomputed embeddings from scratch cache...
Embedding matrix: (6497, 384) computed in 256.73s on cpu


Synopses exceeding 256 tokens: 237 / 6,497 (3.65%)


In [3]:
def search_tfidf(query_idx, k=3, mask=None):
    sims = cosine_similarity(tfidf_mat[query_idx], tfidf_mat)[0]
    if mask is not None:
        sims[~mask] = -np.inf
    else:
        sims[query_idx] = -np.inf
    order = np.lexsort((np.arange(len(sims)), -sims))
    top_indices = order[:k]
    return top_indices, sims[top_indices]

def search_emb(query_idx, k=3, mask=None):
    q_vec = embeddings[query_idx]
    sims = embeddings @ q_vec
    if mask is not None:
        sims[~mask] = -np.inf
    else:
        sims[query_idx] = -np.inf
    order = np.lexsort((np.arange(len(sims)), -sims))
    top_indices = order[:k]
    return top_indices, sims[top_indices]

queries = [
    "Squid Game (2021)",
    "Crash Landing on You (2019)",
    "Alice in Borderland Season 2 (2022)"
]

print("=== Unconstrained Top 3 Neighbors (Self-Excluded Only) ===")
for title in queries:
    m = corpus_df[corpus_df['name'] == title]
    if len(m) == 0:
        continue
    q_idx = m.index[0]
    q_row = corpus_df.iloc[q_idx]
    
    print("=" * 70)
    print(f"QUERY: {q_row['name']} ({q_row['release_year']}, {q_row['country']}) [Tier: {q_row['target_tier']}]")
    print(f"Genres: {q_row['genres']}")
    
    print("\n--- TF-IDF Top 3 ---")
    tf_idx, tf_scores = search_tfidf(q_idx, k=3)
    for rank, (idx, score) in enumerate(zip(tf_idx, tf_scores), 1):
        r = corpus_df.iloc[idx]
        print(f"  {rank}. {r['name']} ({r['release_year']}, {r['country']}) - Score: {score:.3f} [Tier: {r['target_tier']}]")
        print(f"     Genres: {r['genres']}")
        
    print("\n--- Sentence Embeddings Top 3 ---")
    emb_idx, emb_scores = search_emb(q_idx, k=3)
    for rank, (idx, score) in enumerate(zip(emb_idx, emb_scores), 1):
        r = corpus_df.iloc[idx]
        print(f"  {rank}. {r['name']} ({r['release_year']}, {r['country']}) - Score: {score:.3f} [Tier: {r['target_tier']}]")
        print(f"     Genres: {r['genres']}")
    print()


=== Unconstrained Top 3 Neighbors (Self-Excluded Only) ===
QUERY: Squid Game (2021) (2021, South Korea) [Tier: High]
Genres: Action | Thriller | Mystery | Drama

--- TF-IDF Top 3 ---
  1. Squid Game Season 2 (2024) (2024, South Korea) - Score: 0.147 [Tier: High]
     Genres: Action | Thriller | Mystery | Drama
  2. Final Fantasy XIV: Hikari no Otousan (2017) (2017, Japan) - Score: 0.145 [Tier: High]
     Genres: Life | Family
  3. The Romance of Summer (2021) (2021, China) - Score: 0.138 [Tier: Medium]
     Genres: Comedy | Romance | Youth

--- Sentence Embeddings Top 3 ---
  1. Squid Game Season 2 (2024) (2024, South Korea) - Score: 0.650 [Tier: High]
     Genres: Action | Thriller | Mystery | Drama
  2. Last (2015) (2015, South Korea) - Score: 0.590 [Tier: Medium]
     Genres: Action | Thriller | Crime
  3. Cinderella at 2 AM (2024) (2024, South Korea) - Score: 0.578 [Tier: High]
     Genres: Comedy | Romance

QUERY: Crash Landing on You (2019) (2019, South Korea) [Tier: High]
Genres

### Qualitative Observations (Printed Fields Only)

Three hand-picked queries are anecdotes:
1. **Crash Landing on You (2019)**: The printed fields show that TF-IDF retrieved dramas with explicit Korean peninsula themes (*K-Idol Invasion*, *Windy Mi Poong*, *My Military Valentine*), driven by vocabulary overlap on terms like "North Korea". In contrast, the sentence embedding top 3 (*Wanted*, *Mrs. Cop Season 2*, *The Elegant Empire*) appeared generic from the genre metadata alone.
2. **Squid Game (2021)**: The sentence embedding top result is its own sequel (*Squid Game Season 2 (2024)*, score 0.650), whereas TF-IDF matched *Squid Game Season 2* at score 0.147 along with *Final Fantasy XIV: Hikari no Otousan* (0.145) and *The Romance of Summer* (0.138).
3. **Score Comparability**: The similarity scores of the two methods are not on the same scale and cannot be directly compared (TF-IDF scores are sparse cosine products ranging ~0.1–0.4, whereas dense normalized embedding dot products range ~0.4–0.7).
4. **Generalization**: Anecdotal examples cannot establish which method is superior; systematic evaluation is performed using genre overlap Jaccard and franchise retrieval metrics across the full corpus.


In [4]:
print("=== Maturity-Filtered Top 3 Neighbors (start_date <= query_start - 365 days) ===")
for title in queries:
    m = corpus_df[corpus_df['name'] == title]
    if len(m) == 0:
        continue
    q_idx = m.index[0]
    q_row = corpus_df.iloc[q_idx]
    cutoff = q_row['eff_date'] - pd.Timedelta(days=365)
    
    mask = (corpus_df['eff_date'] <= cutoff) & (corpus_df.index != q_idx)
    n_candidates = mask.sum()
    
    print("=" * 70)
    print(f"QUERY: {q_row['name']} ({q_row['release_year']}, {q_row['country']}) [Tier: {q_row['target_tier']}]")
    print(f"Start: {q_row['eff_date'].strftime('%Y-%m-%d')} | Cutoff (<= -365d): {cutoff.strftime('%Y-%m-%d')} | Candidates: {n_candidates:,}")
    print(f"Genres: {q_row['genres']}")
    
    print("\n--- TF-IDF Top 3 (Maturity Filtered) ---")
    tf_idx, tf_scores = search_tfidf(q_idx, k=3, mask=mask)
    for rank, (idx, score) in enumerate(zip(tf_idx, tf_scores), 1):
        r = corpus_df.iloc[idx]
        print(f"  {rank}. {r['name']} ({r['release_year']}, {r['country']}) - Score: {score:.3f} [Tier: {r['target_tier']}]")
        print(f"     Genres: {r['genres']}")
        
    print("\n--- Sentence Embeddings Top 3 (Maturity Filtered) ---")
    emb_idx, emb_scores = search_emb(q_idx, k=3, mask=mask)
    for rank, (idx, score) in enumerate(zip(emb_idx, emb_scores), 1):
        r = corpus_df.iloc[idx]
        print(f"  {rank}. {r['name']} ({r['release_year']}, {r['country']}) - Score: {score:.3f} [Tier: {r['target_tier']}]")
        print(f"     Genres: {r['genres']}")
    print()


=== Maturity-Filtered Top 3 Neighbors (start_date <= query_start - 365 days) ===
QUERY: Squid Game (2021) (2021, South Korea) [Tier: High]
Start: 2021-09-17 | Cutoff (<= -365d): 2020-09-17 | Candidates: 3,088
Genres: Action | Thriller | Mystery | Drama

--- TF-IDF Top 3 (Maturity Filtered) ---
  1. Final Fantasy XIV: Hikari no Otousan (2017) (2017, Japan) - Score: 0.145 [Tier: High]
     Genres: Life | Family
  2. My Mister (2018) (2018, South Korea) - Score: 0.131 [Tier: High]
     Genres: Psychological | Life | Drama
  3. Teacher Oh Soon Nam (2017) (2017, South Korea) - Score: 0.124 [Tier: Medium]
     Genres: Life | Youth | Drama | Melodrama

--- Sentence Embeddings Top 3 (Maturity Filtered) ---
  1. Last (2015) (2015, South Korea) - Score: 0.590 [Tier: Medium]
     Genres: Action | Thriller | Crime
  2. Reply 1988 (2015) (2015, South Korea) - Score: 0.548 [Tier: High]
     Genres: Comedy | Romance | Life | Youth
  3. Woman of 9.9 Billion (2019) (2019, South Korea) - Score: 0.526 [T

## Step 6: Systematic Evaluation on Test Queries (2022–2025)

We systematically evaluate the retrieval methods across all test set dramas with valid synopses ($N = 2,252$, released 2022 or later).
Candidate pools enforce the 12-month maturity rule ($start\_date \le query\_start - 365\text{ days}$) and self-exclusion.


In [5]:
eval_cache_path = scratch_dir / "retriever_eval_df.parquet"

if eval_cache_path.exists():
    eval_df = pd.read_parquet(eval_cache_path)
    print(f"Loaded cached evaluation metrics for N = {len(eval_df):,} test dramas.")
else:
    raise FileNotFoundError("retriever_eval_df.parquet not found in scratch directory.")

def bootstrap_ci(arr, n_boot=500, seed=42):
    rng = np.random.RandomState(seed)
    n = len(arr)
    m = float(np.mean(arr))
    boot_m = [np.mean(rng.choice(arr, size=n, replace=True)) for _ in range(n_boot)]
    return m, float(np.percentile(boot_m, 2.5)), float(np.percentile(boot_m, 97.5))

def paired_diff(arr_a, arr_b, n_boot=500, seed=42):
    rng = np.random.RandomState(seed)
    n = len(arr_a)
    diff = float(np.mean(arr_a) - np.mean(arr_b))
    diffs = [np.mean(arr_a[idx]) - np.mean(arr_b[idx]) for idx in [rng.choice(n, size=n, replace=True) for _ in range(n_boot)]]
    return diff, float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))

print("\n=== Step 6(a): Genre Jaccard & Franchise Matches ===")
print("Method          | Jaccard (All)            | Jaccard (No Franchise)   | Franchise Top-5")
print("-" * 78)
for label, col, col_nf, col_fr in [
    ("Random Pool", "r_jac", "r_jac_nf", "r_fr"),
    ("TF-IDF", "tf_jac", "tf_jac_nf", "tf_fr"),
    ("Embeddings", "emb_jac", "emb_jac_nf", "emb_fr"),
]:
    m, lo, hi = bootstrap_ci(eval_df[col].values)
    mnf, lonf, hinf = bootstrap_ci(eval_df[col_nf].dropna().values)
    fr = np.mean(eval_df[col_fr].values) * 100
    print(f"{label:<15} | {m:.3f} [{lo:.3f}, {hi:.3f}]      | {mnf:.3f} [{lonf:.3f}, {hinf:.3f}]      | {fr:.2f}%")

print("\n=== Paired Bootstrap Differences (Embeddings - TF-IDF, TF-IDF - Random) ===")
print("Slice / Country      | N      | Emb minus TF-IDF [95% CI] | TF-IDF minus Rand [95% CI]")
print("-" * 75)
d_et, l_et, h_et = paired_diff(eval_df["emb_jac"].values, eval_df["tf_jac"].values)
d_tr, l_tr, h_tr = paired_diff(eval_df["tf_jac"].values, eval_df["r_jac"].values)
print(f"Overall (All Test)   | {len(eval_df):<6} | +{d_et:.3f} [+{l_et:.3f}, +{h_et:.3f}]      | +{d_tr:.3f} [+{l_tr:.3f}, +{h_tr:.3f}]")

for country in ["South Korea", "China", "Japan", "Thailand"]:
    sub = eval_df[eval_df["country"] == country]
    det, let, het = paired_diff(sub["emb_jac"].values, sub["tf_jac"].values)
    dtr, ltr, htr = paired_diff(sub["tf_jac"].values, sub["r_jac"].values)
    print(f"{country:<20} | {len(sub):<6} | +{det:.3f} [+{let:.3f}, +{het:.3f}]      | +{dtr:.3f} [+{ltr:.3f}, +{htr:.3f}]")


Loaded cached evaluation metrics for N = 2,252 test dramas.

=== Step 6(a): Genre Jaccard & Franchise Matches ===
Method          | Jaccard (All)            | Jaccard (No Franchise)   | Franchise Top-5
------------------------------------------------------------------------------


Random Pool     | 0.205 [0.200, 0.211]      | 0.205 [0.200, 0.210]      | 0.00%


TF-IDF          | 0.277 [0.270, 0.283]      | 0.275 [0.269, 0.281]      | 0.35%
Embeddings      | 0.306 [0.300, 0.312]      | 0.305 [0.299, 0.311]      | 0.28%

=== Paired Bootstrap Differences (Embeddings - TF-IDF, TF-IDF - Random) ===
Slice / Country      | N      | Emb minus TF-IDF [95% CI] | TF-IDF minus Rand [95% CI]
---------------------------------------------------------------------------


Overall (All Test)   | 2252   | +0.029 [+0.023, +0.035]      | +0.072 [+0.065, +0.077]
South Korea          | 690    | +0.034 [+0.024, +0.043]      | +0.070 [+0.060, +0.080]


China                | 420    | +0.022 [+0.010, +0.036]      | +0.066 [+0.053, +0.078]
Japan                | 401    | +0.035 [+0.020, +0.048]      | +0.079 [+0.064, +0.093]
Thailand             | 741    | +0.026 [+0.015, +0.037]      | +0.072 [+0.062, +0.084]


In [6]:
print("=== Step 6(c): Outcome Diagnostic (Diagnostic only; not evidence of success prediction) ===")

def bootstrap_spearman(x, y, n_boot=500, seed=42):
    rng = np.random.RandomState(seed)
    n = len(x)
    base_rho, _ = spearmanr(x, y)
    rhos = [spearmanr(x[idx], y[idx])[0] for idx in [rng.choice(n, size=n, replace=True) for _ in range(n_boot)]]
    return float(base_rho), float(np.percentile(rhos, 2.5)), float(np.percentile(rhos, 97.5))

def paired_spearman_diffs(sub_df, n_boot=500, seed=42):
    rng = np.random.RandomState(seed)
    n = len(sub_df)
    y = sub_df["target_pct"].values
    re_v = sub_df["emb_pct10"].values
    rt_v = sub_df["tf_pct10"].values
    rg_v = sub_df["g_pct10"].values

    base_re, _ = spearmanr(y, re_v)
    base_rt, _ = spearmanr(y, rt_v)
    base_rg, _ = spearmanr(y, rg_v)

    d_et_list, d_eg_list, d_gt_list = [], [], []
    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        yb = y[idx]
        re_b, _ = spearmanr(yb, re_v[idx])
        rt_b, _ = spearmanr(yb, rt_v[idx])
        rg_b, _ = spearmanr(yb, rg_v[idx])
        d_et_list.append(re_b - rt_b)
        d_eg_list.append(re_b - rg_b)
        d_gt_list.append(rg_b - rt_b)

    return {
        "et": (base_re - base_rt, np.percentile(d_et_list, 2.5), np.percentile(d_et_list, 97.5)),
        "eg": (base_re - base_rg, np.percentile(d_eg_list, 2.5), np.percentile(d_eg_list, 97.5)),
        "gt": (base_rg - base_rt, np.percentile(d_gt_list, 2.5), np.percentile(d_gt_list, 97.5)),
    }

for name, sub in [
    ("All Test Queries", eval_df),
    ("complete_info (N=997)", eval_df[eval_df["complete_info"] == True])
]:
    y = sub["target_pct"].values
    print(f"\n--- Spearman rho: Mean Top-10 target_pct vs Query target_pct ({name}, N = {len(sub):,}) ---")
    for m_label, col in [
        ("Random Pool", "r_pct10"),
        ("Genre-Only Retriever", "g_pct10"),
        ("TF-IDF", "tf_pct10"),
        ("Embeddings", "emb_pct10")
    ]:
        rho, lo, hi = bootstrap_spearman(y, sub[col].values)
        sign = "+" if rho >= 0 else ""
        print(f"  {m_label:<25} | {sign}{rho:.3f} [{lo:+.3f}, {hi:+.3f}]")

    diffs = paired_spearman_diffs(sub)
    print("  Paired Differences:")
    print(f"    Embeddings minus TF-IDF:     {diffs['et'][0]:+.3f} [{diffs['et'][1]:+.3f}, {diffs['et'][2]:+.3f}]")
    print(f"    Embeddings minus Genre-Only: {diffs['eg'][0]:+.3f} [{diffs['eg'][1]:+.3f}, {diffs['eg'][2]:+.3f}]")
    print(f"    Genre-Only minus TF-IDF:     {diffs['gt'][0]:+.3f} [{diffs['gt'][1]:+.3f}, {diffs['gt'][2]:+.3f}]")

print("\n=== High-Tier Queries (N = 638) - Share of Top-5 in High Tier (Selects on Outcome) ===")
high_sub = eval_df[eval_df["target_tier"] == "High"]
for l, c in [
    ("Pool Base Rate", "pool_high"),
    ("Random Neighbors", "r_high"),
    ("Genre-Only Retriever", "g_high"),
    ("TF-IDF Top-5", "tf_high"),
    ("Embeddings Top-5", "emb_high"),
]:
    m, lo, hi = bootstrap_ci(high_sub[c].values)
    print(f"  {l:<25} | {m*100:.1f}% [{lo*100:.1f}%, {hi*100:.1f}%]")

print("\n=== Low-Tier Queries (N = 404) - Share of Top-5 in Low Tier (Selects on Outcome) ===")
low_sub = eval_df[eval_df["target_tier"] == "Low"]
for l, c in [
    ("Pool Base Rate", "pool_low"),
    ("Random Neighbors", "r_low"),
    ("Genre-Only Retriever", "g_low"),
    ("TF-IDF Top-5", "tf_low"),
    ("Embeddings Top-5", "emb_low"),
]:
    m, lo, hi = bootstrap_ci(low_sub[c].values)
    print(f"  {l:<25} | {m*100:.1f}% [{lo*100:.1f}%, {hi*100:.1f}%]")

print("\n=== Selection Bias on Synopsis Availability across All 7,535 Labeled Dramas ===")
df_lab_all = pd.read_csv(data_dir / "dramas_labeled.csv")
df_syn_all = pd.read_csv(data_dir / "synopses_clean.csv")
m_all = pd.merge(df_lab_all, df_syn_all[["url", "synopsis_clean"]], on="url")
has_syn = m_all["synopsis_clean"].astype(str).str.strip().str.len() > 0

print(f"With Synopsis (N = {has_syn.sum():,}):")
print(f"  High: {(m_all.loc[has_syn, 'target_tier'] == 'High').mean()*100:.2f}% | Low: {(m_all.loc[has_syn, 'target_tier'] == 'Low').mean()*100:.2f}%")
print(f"Without Synopsis (N = {(~has_syn).sum():,}):")
print(f"  High: {(m_all.loc[~has_syn, 'target_tier'] == 'High').mean()*100:.2f}% | Low: {(m_all.loc[~has_syn, 'target_tier'] == 'Low').mean()*100:.2f}%")

# Multi-seed random baseline summary across 100 seeds
rand_summary_path = scratch_dir / "random_100_seeds_summary.json"
if rand_summary_path.exists():
    import json
    with open(rand_summary_path, "r", encoding="utf-8") as f:
        r_sum = json.load(f)
    print("\n=== Random Baseline across 100 Seeds (Mean and 2.5-97.5% Spread) ===")
    print(f"Genre Jaccard (All Top-5):         {r_sum['jaccard_all']['mean']:.3f} [{r_sum['jaccard_all']['lo']:.3f}, {r_sum['jaccard_all']['hi']:.3f}]")
    print(f"Genre Jaccard (No Franchise):      {r_sum['jaccard_nof']['mean']:.3f} [{r_sum['jaccard_nof']['lo']:.3f}, {r_sum['jaccard_nof']['hi']:.3f}]")
    print(f"TF-IDF minus Random (Overall):     +{r_sum['diff_overall']['mean']:.3f} [+{r_sum['diff_overall']['lo']:.3f}, +{r_sum['diff_overall']['hi']:.3f}]")
    print(f"TF-IDF minus Random (South Korea): +{r_sum['diff_korea']['mean']:.3f} [+{r_sum['diff_korea']['lo']:.3f}, +{r_sum['diff_korea']['hi']:.3f}]")
    print(f"TF-IDF minus Random (China):       +{r_sum['diff_china']['mean']:.3f} [+{r_sum['diff_china']['lo']:.3f}, +{r_sum['diff_china']['hi']:.3f}]")
    print(f"TF-IDF minus Random (Japan):       +{r_sum['diff_japan']['mean']:.3f} [+{r_sum['diff_japan']['lo']:.3f}, +{r_sum['diff_japan']['hi']:.3f}]")
    print(f"TF-IDF minus Random (Thailand):    +{r_sum['diff_thai']['mean']:.3f} [+{r_sum['diff_thai']['lo']:.3f}, +{r_sum['diff_thai']['hi']:.3f}]")
    print(f"Spearman rho (All Test Queries):   {r_sum['spearman_all']['mean']:+.3f} [{r_sum['spearman_all']['lo']:+.3f}, {r_sum['spearman_all']['hi']:+.3f}]")
    print(f"Spearman rho (complete_info):      {r_sum['spearman_comp']['mean']:+.3f} [{r_sum['spearman_comp']['lo']:+.3f}, {r_sum['spearman_comp']['hi']:+.3f}]")
    print(f"High-Tier Queries Top-5 High:      {r_sum['high_share']['mean']*100:.1f}% [{r_sum['high_share']['lo']*100:.1f}%, {r_sum['high_share']['hi']*100:.1f}%]")
    print(f"Low-Tier Queries Top-5 Low:        {r_sum['low_share']['mean']*100:.1f}% [{r_sum['low_share']['lo']*100:.1f}%, {r_sum['low_share']['hi']*100:.1f}%]")


=== Step 6(c): Outcome Diagnostic (Diagnostic only; not evidence of success prediction) ===

--- Spearman rho: Mean Top-10 target_pct vs Query target_pct (All Test Queries, N = 2,252) ---


  Random Pool               | -0.045 [-0.085, -0.003]


  Genre-Only Retriever      | +0.356 [+0.324, +0.395]


  TF-IDF                    | +0.251 [+0.211, +0.296]


  Embeddings                | +0.291 [+0.253, +0.330]


...

  Genre-Only Retriever      | +0.317 [+0.259, +0.372]


  TF-IDF                    | +0.258 [+0.197, +0.313]


  Embeddings                | +0.293 [+0.232, +0.349]


  Paired Differences:
    Embeddings minus TF-IDF:     +0.035 [-0.024, +0.100]
    Embeddings minus Genre-Only: -0.024 [-0.095, +0.052]
    Genre-Only minus TF-IDF:     +0.059 [-0.011, +0.139]

=== High-Tier Queries (N = 638) - Share of Top-5 in High Tier (Selects on Outcome) ===
  Pool Base Rate            | 29.8% [29.8%, 29.8%]
  Random Neighbors          | 28.8% [27.2%, 30.6%]
  Genre-Only Retriever      | 39.8% [37.9%, 41.7%]
  TF-IDF Top-5              | 33.6% [31.9%, 35.3%]
  Embeddings Top-5          | 40.9% [39.1%, 42.8%]

=== Low-Tier Queries (N = 404) - Share of Top-5 in Low Tier (Selects on Outcome) ===
  Pool Base Rate            | 15.9% [15.8%, 15.9%]
  Random Neighbors          | 16.5% [15.0%, 18.1%]
  Genre-Only Retriever      | 29.0% [26.4%, 31.6%]
  TF-IDF Top-5              | 20.9% [18.9%, 22.9%]
  Embeddings Top-5          | 19.1% [17.0%, 20.9%]

=== Selection Bias on Synopsis Availability across All 7,535 Labeled Dramas ===


With Synopsis (N = 6,497):
  High: 29.29% | Low: 16.52%
Without Synopsis (N = 1,038):
  High: 0.19% | Low: 76.30%

=== Random Baseline across 100 Seeds (Mean and 2.5-97.5% Spread) ===
Genre Jaccard (All Top-5):         0.205 [0.201, 0.208]
Genre Jaccard (No Franchise):      0.205 [0.201, 0.208]
TF-IDF minus Random (Overall):     +0.072 [+0.069, +0.076]
TF-IDF minus Random (South Korea): +0.068 [+0.062, +0.073]
TF-IDF minus Random (China):       +0.065 [+0.058, +0.072]
TF-IDF minus Random (Japan):       +0.080 [+0.073, +0.087]
TF-IDF minus Random (Thailand):    +0.076 [+0.069, +0.080]
Spearman rho (All Test Queries):   +0.003 [-0.043, +0.052]
Spearman rho (complete_info):      -0.003 [-0.064, +0.064]
High-Tier Queries Top-5 High:      29.8% [28.3%, 31.2%]
Low-Tier Queries Top-5 Low:        15.9% [14.6%, 17.4%]


## Human-Judged Precision@5 (Pending)

> **Status: PENDING LABELING (Single Labeler)**
> 
> A blind sample of 30 randomly selected test dramas (split into three 10-query parts: `retriever_blind_labeling_part1.csv`, `part2.csv`, `part3.csv`) contains 300 blinded query-neighbor pairs (including 10 consistency check duplicates).
>
> - **Evaluation Rubric**: Relevant ($1$) = the central premise, dramatic situation, or core conflict is similar (e.g., deadly survival game, workplace romance between rivals, parallel timeline murder mystery); sharing only a genre tag or country is $0$.
> - Precision@5 per method (TF-IDF vs. Sentence Embeddings) will be computed from the submitted labels via the answer key.
